# Tribunal de Justiça de São Paulo (TJSP)

[![Repo](https://img.shields.io/badge/GitHub-repo-blue?logo=github&logoColor=f5f5f5)](https://github.com/open-geodata/sp_tjsp_divadmin)

Necessário dar continuidade.


In [ ]:
import concurrent.futures

import pandas as pd
from dotenv import load_dotenv

import open_geodata as geo

<br>

---

## Regiões Administrativas Judiciárias (RAJs)

Inicialmente, com auxílio do do pacote [lxml](https://lxml.de/), acessamos as informações da página [Quem Somos](https://www.tjsp.jus.br/QuemSomos/QuemSomos/RegioesAdministrativasJudiciarias) do TJSP.


In [ ]:
tjsp = geo.sp.tjsp.div_admin.QuemSomos()
df_rajs = tjsp.get_raj()
df_rajs.info()
df_rajs.head()

<br>

Além das **_RAJs_**, obtemos também todas as comarcas, bem como qual a **_Circunscrição Judiciária_** que abrange aquela comarca.


In [ ]:
df_comarcas = tjsp.get_comarcas()

# Ajusta Coluna
df_script1 = geo.sp.tjsp.div_admin.adjust_columns(
    df=df_comarcas,
    column_ajust='comarca_tjsp',
)

# Resultados
df_comarcas.info()
df_comarcas.head()

<br>

Já é possível relacionar as **_Circunscrição Judiciária_** às **_RAJs_**.


In [ ]:
df_cj = tjsp.get_cj()
df_cj.info()
df_cj.head()

<br>

Agora só falta definir quais os municípios que pertencem a qual comarca.


<br>

---

## Lista Telefônica do TJSP

O TJSP tem tantas unicades espalhadas pelo Estado de São Paulo que há uma página dedicada à [Lista Telefônica](https://www.tjsp.jus.br/ListaTelefonica), onde digitando (a partir de) 3 caracteres, a página retorna 10 municípios sugeridos.


In [ ]:
tjsp = geo.sp.tjsp.div_admin.TJSP()
tjsp.get_lista_municipios_tjsp(contain='San')

<br>

Como não sei como o TJSP escreve o nome dos municípios, optei por pegar o nome de todos os municípios do estado de São Paulo (com a grafia que eu ajustei) e iterar, a partir de 3 caracteres, a pesquisa.

Por exemplo, o município de "Piracicaba" será pesquisado a partir do 3º caractere.

- Pir
- Pira
- Pirac
- Piraci
- Piracic
- Piracica
- Piracicab
- Piracicaba

<br>

Para cada pesquisa será retornado os 10 municípios com a grafia mais semelhante e, por fim, obtenho os 645 municípios do Estado de São Paulo.


In [ ]:
list_termos = tjsp.list_terms_search()

<br>

Para agilizar a pesquisa de mais de 4.000 termos (fragmentos de nomes de municípios), optei por usar o pacote [requests_ip_rotator](https://github.com/Ge0rg3/requests-ip-rotator), que instancia uma API na [AWS](https://aws.amazon.com/) para disparar desenas de requisições simultâneas.


In [ ]:
load_dotenv()

AWS_ACCESS_KEY_ID = os.getenv('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = os.getenv('AWS_SECRET_ACCESS_KEY')

In [ ]:
df = tjsp.search_terms_aws(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

df.info()
df.head()

<br>

Por fim obtenho a lista dos 645 municípios.


In [ ]:
df_mun_tjsp = tjsp.adjust_data(df=df)
df_mun_tjsp.info()
df_mun_tjsp.head()

<br>

Sei da existência de nome errados do TJSP e já corrijo.


In [ ]:
df_mun_tjsp['municipio_tjsp'] = df_mun_tjsp['municipio_tjsp'].replace(
    {
        # Nome Errado (TJSP): Nome Correto (Michel)
        'Estrela dOeste': "Estrela d'Oeste",
        'Luís Antônio': 'Luiz Antônio',
        'Florínia': 'Florínea',
    }
)

<br>

Agora posso dar sequencia da unificação com os códigos do IBGE.

<br>

---

## Municípios TJSP

Inicialmente lemos a tabela que tem o nome correto dos municípios (eu que defini o que é correto ou não, acessando sites de prefeituras).


In [ ]:
df_mun_sp = geo.data.load_dataset(db='sp', name='tab.municipio_nome')
df_mun_sp.head()

<br>

E concatenamos a tabela do TJSP com a tabela contendo o nome dos municípios.

In [ ]:
df_mun = pd.merge(
    left=df_mun_sp,
    right=df_mun_tjsp,
    left_on='municipio_nome',
    right_on='municipio_tjsp',
    how='left',
)

# Confere se há algum erro
df_temp = df_mun[df_mun['municipio_tjsp'].isnull()]
if len(df_temp):
    raise Exception('Deu ruim')

# Results
df_mun.info()
df_mun.head()

In [ ]:
df_mun.to_csv(
    'tjsp_municipios.csv',
    index=False,
    encoding='utf-8',
)

In [ ]:
df_mun = pd.read_csv('tjsp_municipios.csv')
df_mun

<br>


-----

## Unidades

Ainda usando a Lista Telefônica do TJSP, encontrei uma função que é possível pesquisar pelo código (do TJSP) do município. Defini uma função que me retorna uma tabela contendo todas as unidades do TJSP (Fóruns, Setores, prédios etc) e, ainda, me informa a qual comarca o município pertence.

In [ ]:
lista = geo.providers.sp.tjsp.div_admin.ListaTelefonica()
lista.get_lista_unidades_tjsp(cod_municipio=6504)

In [ ]:
# Lista Municípios
list_cod_tjsp = list(df_mun['id_municipio_tjsp'])
list_cod_tjsp[:5]

<br>

Iteramos as requisições dos 645 municípios.

In [ ]:
MAX_THREADS = 5

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
    temp = executor.map(lista.get_lista_unidades_tjsp, list_cod_tjsp)
    df_com = pd.concat(list(temp), ignore_index=True)

# Results
df_com.info()
df_com.head()

<br>

Montamos uma tabela contendo tudo.

In [ ]:
# Merge
df_unidades = pd.merge(
    left=df_mun,
    right=df_com,
    left_on='id_municipio_tjsp',
    right_on='id_municipio_tjsp',
    how='inner',
    suffixes=['', '_copy'],
)

# Exclui coluna repetida
df_unidades = df_unidades.drop(
    labels=['municipio_tjsp_copy'],
    axis='columns',
    errors='ignore',
)

# Aplica strip em todo o dataframe
df_unidades = df_unidades.map(lambda x: x.strip() if isinstance(x, str) else x)

# Results
df_unidades.info()
df_unidades.head()

<br>

-----

## Municípios

In [ ]:
df_mun_com = df_unidades[
    [
        'id_municipio',
        'id_municipio_tjsp',
        'municipio_nome',
        'municipio_tjsp',
        #'raj',
        'comarca_tjsp',
        'comarca_sede',
        #'unidades',
    ]
]
df_mun_com = df_mun_com.drop_duplicates()
df_mun_com = df_mun_com.reset_index(drop=True)
df_mun_com.info()
df_mun_com.head()

<br>

Crio uma tabela temporária, apenas para conseguir, posteriormente, trazer os códigos do IBGE para a tabela das comarcas


In [ ]:
# Comarca
df_mun_com_sede = df_mun_com[df_mun_com['comarca_sede'] == 1]

# # Renomeia
# df_mun_com_sede = df_mun_com_sede.rename(
#     mapper={
#         'municipio_nome': 'comcarca_nome',
#         'id_municipio': 'id_comarca',
#     },
#     axis=1,
# )

# # Deleta Coluna
# df_mun_com_sede = df_mun_com_sede.drop(
#     labels=['comarca_sede', 'municipio_tjsp', 'comarca_tjsp'],
#     axis='columns',
#     errors='ignore',
# )

# Deleta Duplicados
df_mun_com_sede = df_mun_com_sede.drop_duplicates()

# Reset Index
df_mun_com_sede = df_mun_com_sede.reset_index(drop=True)

# Bata Bater
# df_tjsp_com = adjust_columns(df=df_tjsp_com, column_ajust='comarca_tjsp')

# Ajusta Coluna
df_mun_com_sede = geo.sp.tjsp.div_admin.adjust_columns(
    df=df_mun_com_sede, column_ajust='comarca_tjsp'
)

# Results
df_mun_com_sede.info()
df_mun_com_sede.head()

In [ ]:
df_mun_com_sede['municipio_tjsp'].equals(df_mun_com_sede['comarca_tjsp'])
df_mun_com_sede[df_mun_com_sede['municipio_tjsp'] != df_mun_com_sede['comarca_tjsp']]